# 03 · What students asked about

Every session gets one of nine labels. Two classifiers are available:

* `keyword` — transparent, deterministic, free. The default, and what a mock run uses.
* `openai` — one `gpt-4o-mini` call per session, the same approach `analytics.ipynb` has always used.
  Set `METHOD = "openai"` below and export `OPENAI_API_KEY` to use it.

**A classifier is not ground truth.** The last cells draw a hand-audit sample and compute the
agreement rate that the report publishes beside every topic chart. A topic chart without that number
is an assertion; with it, it is a measurement.

In [1]:
# Put the analysis package on the path no matter where Jupyter was started.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "analysis" / "bloombot_analysis").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "analysis"))

import pandas as pd

from bloombot_analysis import charts, load, metrics, privacy, report, sessions, topics
from bloombot_analysis.config import CONFIG, SURFACE_LABELS, TOPICS

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("as of:", CONFIG.as_of)
print("legacy db :", CONFIG.legacy_db, "(exists)" if CONFIG.legacy_db.exists() else "(missing)")
print("current db:", CONFIG.current_db, "(exists)" if CONFIG.current_db.exists() else "(missing)")
print("output    :", CONFIG.out_dir)

as of: 2026-09-25
legacy db : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/legacy.db (exists)
current db: /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/current.db (exists)
output    : /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out


In [2]:
sessions_df = pd.read_csv(
    CONFIG.data_path("sessions.csv"), parse_dates=["started_at", "ended_at", "week"]
)
sessions_df["date"] = pd.to_datetime(sessions_df["date"]).dt.date
messages_df = pd.read_csv(CONFIG.data_path("messages.csv"), parse_dates=["ts", "week"])
print(len(sessions_df), "sessions,", len(messages_df), "messages")

259 sessions, 1584 messages


In [3]:
import os

# Switch to "openai" for the real run if a key is available; the keyword
# classifier is the fallback either way, and which one produced each label is
# recorded per session in `topic_method`.
METHOD = os.environ.get("BLOOMBOT_TOPIC_METHOD", "keyword")

tagged = sessions.sessionize(messages_df)
texts = topics.session_texts(tagged)
labelled = topics.classify_sessions(texts, method=METHOD)
labelled = labelled.merge(
    sessions_df[["session_id", "course", "surface", "started_at", "prompts", "messages"]],
    on="session_id",
    how="inner",
    suffixes=("", "_session"),
)
print(labelled["topic_method"].value_counts().to_dict())
labelled["topic"].value_counts()

{'keyword-cached': 259}


topic
Team projects & collaboration     64
Course material & content         36
Assignments & homework            32
Technical setup & tools           29
Resources & references            26
Other                             21
Grades & assessment               20
Professor & office hours          17
Syllabus, schedule & deadlines    14
Name: count, dtype: int64

## Topic mix, all courses

Sorted by volume, single hue: this is magnitude, not identity, so it takes one colour rather than
nine. A large "Other" share is itself a finding about what the label set misses.

In [4]:
counts = topics.topic_counts(labelled).sort_values("sessions")
topic_fig = charts.bar_h(
    counts["topic"], counts["sessions"], "topics_overall", "Sessions by topic — all courses", "Sessions"
)
counts.sort_values("sessions", ascending=False)

,topic,sessions
6,Team projects & collaboration,64
0,Course material & content,36
1,Assignments & homework,32
3,Technical setup & tools,29
7,Resources & references,26
8,Other,21
4,Grades & assessment,20
5,Professor & office hours,17
2,"Syllabus, schedule & deadlines",14


## Topic by course

Cells covering fewer than `CONFIG.min_cell_students` distinct students are suppressed: in a cohort
this small, a cell of one is effectively a named individual. Suppressed cells are drawn empty, never
as zero.

In [5]:
topic_course = topics.topic_by(labelled, "course").T
student_counts = (
    labelled.groupby(["course", "topic"])["session_id"]
    .count()
    .unstack(fill_value=0)
    .reindex(columns=TOPICS, fill_value=0)
    .T
)
# The suppression test counts distinct students, not sessions.
students_behind = (
    labelled.merge(sessions_df[["session_id", "person_key"]], on="session_id", how="left")
    .groupby(["course", "topic"])["person_key"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=TOPICS, fill_value=0)
    .T
)
published = privacy.suppress_small_cells(topic_course.astype(float), students_behind)
note = privacy.suppression_note(topic_course.astype(float), published)
topic_course_fig = charts.heatmap(
    published, "topics_by_course", "Sessions by topic and course", "Sessions"
)
print(note or "no cells suppressed")
published

24 of 36 cells are suppressed: fewer than 5 distinct students behind them.


course,Agile Software Development & DevOps,Introduction to Programming,Software Engineering,Web Design
topic,,,,
Course material & content,11.0,9.0,NaN,NaN
Assignments & homework,NaN,9.0,14.0,NaN
"Syllabus, schedule & deadlines",NaN,NaN,NaN,NaN
Technical setup & tools,10.0,NaN,11.0,NaN
Grades & assessment,NaN,NaN,NaN,NaN
Professor & office hours,NaN,NaN,6.0,NaN
Team projects & collaboration,23.0,13.0,17.0,11.0
Resources & references,NaN,NaN,NaN,NaN
Other,NaN,5.0,NaN,NaN


## This fall against last fall

Same like-for-like window as notebook 01. Read this as suggestive: the new interfaces pulled in
whoever they pulled in, and three weeks of one term is not a behavioural finding.

In [6]:
completeness = sessions.term_completeness()
elapsed = completeness["elapsed_days"]
current_term = CONFIG.term(CONFIG.current_term)
comparison_term = CONFIG.term(CONFIG.comparison_term)

now = sessions.like_for_like(labelled, current_term, elapsed, ts_column="started_at")
then = sessions.like_for_like(labelled, comparison_term, elapsed, ts_column="started_at")
term_topics = pd.DataFrame(
    {
        comparison_term.label: then["topic"].value_counts(),
        current_term.label: now["topic"].value_counts(),
    }
).reindex(TOPICS).fillna(0).astype(int)
term_topics_fig = charts.grouped_bar(
    term_topics.loc[term_topics.sum(axis=1) > 0],
    "topics_by_term",
    f"Topics in the first {elapsed} days of term",
    "Sessions",
)
term_topics

,Fall 2025,Fall 2026
topic,,
Course material & content,3,5
Assignments & homework,4,5
"Syllabus, schedule & deadlines",1,1
Technical setup & tools,4,7
Grades & assessment,0,5
Professor & office hours,2,3
Team projects & collaboration,11,10
Resources & references,2,3
Other,4,2


## Audit sample

Fill in `hand_label` in the CSV below, read it back, and the agreement rate goes on the methodology
slide. The sample is drawn with a fixed seed, so it is the same sample every run.

In [7]:
sample = topics.audit_sample(labelled, n=30)
audit_path = CONFIG.data_path("topic_audit_sample.csv")
sample.to_csv(audit_path, index=False)
print(f"wrote {audit_path} — fill in hand_label, then re-run the cell below")

completed_path = CONFIG.data_path("topic_audit_completed.csv")
if completed_path.exists():
    agreement = topics.agreement_rate(pd.read_csv(completed_path))
else:
    agreement = {"checked": 0, "agreed": 0, "rate": float("nan")}
agreement

wrote /Users/ab1258/Documents/education_automation/bloombot/tmp/analysis/out/data/topic_audit_sample.csv — fill in hand_label, then re-run the cell below


{'checked': 0, 'agreed': 0, 'rate': nan}

## Quotable exchanges

Candidates only: mechanically redacted here, to be read and paraphrased by a human before anything
reaches a slide.

In [8]:
quotes = privacy.quote_candidates(labelled, sessions_df, per_topic=1)
quotes.to_csv(CONFIG.data_path("quote_candidates.csv"), index=False)
quotes.head(10)

,topic,course,quote
0,Professor & office hours,Web Design,Student: when are office hours this week Bot: ...
1,Technical setup & tools,Agile Software Development & DevOps,"Student: docker won't start on port 3000, is t..."
2,Resources & references,Web Design,Student: is there a tutorial you recommend for...
3,Grades & assessment,Web Design,Student: what is the rubric for the midterm Bo...
4,Other,Introduction to Programming,Student: are you a real person Bot: Here's wha...
5,Assignments & homework,Introduction to Programming,Student: do we have to submit the homework as ...
6,"Syllabus, schedule & deadlines",Web Design,Student: can I submit the homework late if I'm...
7,Course material & content,Web Design,Student: can you explain what a closure is Bot...
8,Team projects & collaboration,Agile Software Development & DevOps,Student: how should our group split up the spr...


In [9]:
labelled.drop(columns=["text", "student_text"]).to_csv(
    CONFIG.data_path("session_topics.csv"), index=False
)

metrics.update("topics", {
    "method": METHOD,
    "method_counts": labelled["topic_method"].value_counts().to_dict(),
    "counts": counts.sort_values("sessions", ascending=False).to_dict(orient="records"),
    "top_topic": counts.sort_values("sessions", ascending=False).iloc[0]["topic"],
    "top_topic_sessions": int(counts["sessions"].max()),
    "other_share": float(
        counts.loc[counts["topic"] == "Other", "sessions"].sum() / max(1, counts["sessions"].sum())
    ),
    "by_course": published.to_dict(),
    "suppression_note": note,
    "by_term": term_topics.to_dict(),
    "agreement": agreement,
    "quotes": quotes.to_dict(orient="records"),
    "figures": {
        "overall": topic_fig.name,
        "by_course": topic_course_fig.name,
        "by_term": term_topics_fig.name,
    },
})
print("ok")

ok
